# Notebook 04 - Comparative TSTR Experiments

**Authors:** Ivo Rambaldi & Tommaso Petrelli  
**Course:** Ethics in Artificial Intelligence, University of Bologna, a.y. 2025-2026

---

## Overview

This notebook implements **Goal 4** of the project: we evaluate how replacing real training data with synthetically generated data affects two dimensions that matter for responsible machine learning:

- **Utility**: how accurately can the model predict high academic performance?
- **Fairness**: how equitably are predictions distributed across groups defined by circumstances (socioeconomic status, gender, parental education, etc.) that are beyond a student's control?

The evaluation protocol is **Train on Synthetic, Test on Real (TSTR)**: each classifier is trained on one of the synthetic datasets (or the real training set, as a baseline) and evaluated on the *same* real held-out test set. This setup isolates the causal effect of the training data on downstream behaviour; any change in utility or fairness can be attributed directly to the synthetic generation process, not to the test conditions.

### Experiment grid

| Dimension | Values |
|-----------|--------|
| **Training data** | Real (baseline), Copula, CTGAN, SMOTE, TVAE |
| **Classifiers** | Logistic Regression, XGBoost, MLP |
| **Utility metrics** | Balanced Accuracy, F1-macro, ROC-AUC, Brier Score, MMD |
| **Fairness metrics** | DPD, EOD, DI (per protected attribute and averaged) |

Hyperparameters are fixed across all experiments (see `config/config.yaml`) so that results differences are attributable solely to training data, not to tuning differences.

All outputs (metrics CSVs, figures) are written to `results/experiments/` and `results/figures/experiments/`. The final analysis and interpretation of these results is left to **Notebook 5**.

## 1. Environment Setup

### 1.1 Path and import setup

In [3]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
%matplotlib inline

import numpy as np
import pandas as pd

# Suppress warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Ensure the project root is on sys.path so 'src' imports resolve correctly
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.logging import get_logger


logger = get_logger('notebook_04')

logger.info(f"Environment ready. Project root: {PROJECT_ROOT}")

2026-06-28 15:31:23 | INFO     | notebook_04 | Environment ready. Project root: D:\Users\Utente\OneDrive - Alma Mater Studiorum Università di Bologna\università\Studio\5ANNO\2SEMESTRE\Ethics\Project\Dev\petrellirambaldi2526


### 1.2 Load configuration

In [4]:
from src.utils.config import load_config
from src.data.persistence import ensure_dir


# Load config.yaml
cfg = load_config(PROJECT_ROOT / 'config' / 'config.yaml')

# Resolve output directories and create them if necessary.
EXPERIMENTS_DIR = Path(cfg["paths"]["experiments_dir"])
FIGURES_DIR     = Path(cfg["paths"]["figures_dir"])
ensure_dir(EXPERIMENTS_DIR)
ensure_dir(FIGURES_DIR)

# Print key config values for reproducibility
methods     = cfg["generation"]["methods"]
classifiers = [k for k, v in cfg["experiments"]["classifiers"].items() if v.get("enabled", True)]
protected   = cfg["dataset"]["protected_attributes"]

print('=== Configuration summary ===')
print(f"Seed                : {cfg['seed']}")
print(f"Generation methods  : {methods}")
print(f"Classifiers         : {classifiers}")
print(f"Protected attributes: {len(protected)}")
print(f"Total experiments   : {(len(methods) + 1) * len(classifiers)}")

=== Configuration summary ===
Seed                : 42
Generation methods  : {'ctgan': {'enabled': True, 'epochs': 300, 'batch_size': 500}, 'tvae': {'enabled': True, 'epochs': 300, 'batch_size': 500}, 'gaussian_copula': {'enabled': True}, 'smote': {'enabled': True, 'k_neighbors': 5}}
Classifiers         : ['logistic_regression', 'xgboost', 'mlp']
Protected attributes: 12
Total experiments   : 15


## 2. Data Loading

We load:
1. The **real train / test splits** produced by Notebook 1 (Goal 1). These are the reference datasets.
2. One **synthetic training dataset per generation method** produced by Notebook 2 (Goal 2).

The real test set is never used for training or generation; it serves exclusively as the evaluation target throughout all experiments.

In [5]:
from src.data.loader import load_real_data, load_all_synthetic_datasets


# Load real data (baseline)
real_train_df, real_test_df = load_real_data(cfg)

logger.info(f"Real train: {real_train_df.shape}")
logger.info(f"Real test : {real_test_df.shape}")

# Load synthetic datasets
synthetic_datasets = load_all_synthetic_datasets(cfg)

for method, df in synthetic_datasets.items():
    logger.info(f"Synthetic [{method}]: {df.shape}")

2026-06-28 15:31:23 | INFO     | src.data.loader | Loading real train data from D:\Users\Utente\OneDrive - Alma Mater Studiorum Università di Bologna\università\Studio\5ANNO\2SEMESTRE\Ethics\Project\Dev\petrellirambaldi2526\Data\train.csv
2026-06-28 15:31:23 | INFO     | src.data.loader | Loading real test data  from D:\Users\Utente\OneDrive - Alma Mater Studiorum Università di Bologna\università\Studio\5ANNO\2SEMESTRE\Ethics\Project\Dev\petrellirambaldi2526\Data\test.csv
2026-06-28 15:31:26 | INFO     | notebook_04 | Real train: (58128, 303)
2026-06-28 15:31:26 | INFO     | notebook_04 | Real test : (14533, 303)
2026-06-28 15:31:26 | INFO     | src.data.loader | Loading synthetic data [ctgan] from D:\Users\Utente\OneDrive - Alma Mater Studiorum Università di Bologna\università\Studio\5ANNO\2SEMESTRE\Ethics\Project\Dev\petrellirambaldi2526\results\synthetic\ctgan\synthetic_ctgan.csv


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\Users\\Utente\\OneDrive - Alma Mater Studiorum Università di Bologna\\università\\Studio\\5ANNO\\2SEMESTRE\\Ethics\\Project\\Dev\\petrellirambaldi2526\\results\\synthetic\\ctgan\\synthetic_ctgan.csv'

### 2.1 Target distribution check

Before running experiments, we verify the class distribution in the real test set and across all training datasets. Substantial imbalance in the training sets could confound comparisons between methods unless handled consistently. All classifiers are configured with a `class_weight` or `scale_pos_weight` argument to address this.

In [ ]:
rows = []
for label, df in {
    "real_train": real_train_df,
    "real_test":  real_test_df,
    **synthetic_datasets,
}.items():
    counts = df[cfg['dataset']['target_column']].value_counts(normalize=True)
    rows.append({
        "dataset":       label,
        "n_samples":     len(df),
        "positive_rate": round(counts.get(1, 0.0), 4),
        "negative_rate": round(counts.get(0, 0.0), 4),
    })

dist_df = pd.DataFrame(rows)
display(dist_df)

## 3. Full TSTR Experiment Matrix

We now run every combination of (training dataset, classifier) via a shared
`Evaluator` (`src/evaluation/evaluator.py`), the same class Notebook 03 uses for
its baseline validation. It:

1. Trains the classifier on the given training data (encoded with the same
   OrdinalEncoder + StandardScaler fit on the real training set, so
   LogisticRegression/MLP receive fully numerical input).
2. Evaluates predictions on the real test set.
3. Computes utility metrics, per-attribute and mean fairness metrics, MMD, and
   correlation deltas.

`Evaluator.baseline()` anchors the real-data reference for a given classifier;
`Evaluator.evaluate()` scores a synthetic training set against that same
classifier/reference (TSTR).

In [ ]:
from src.data.preprocessing import get_feature_columns
from src.data.preprocessor import DataSplit, fit_feature_encoders, apply_feature_encoders
from src.evaluation.evaluator import Evaluator
from src.models.classifiers import CLASSIFIER_DISPLAY_NAMES

target_col      = cfg["dataset"]["target_column"]
protected_attrs = cfg["dataset"]["protected_attributes"]

# Pre-compute feature columns from the real training set so all
# experiments use exactly the same feature set.
feature_columns = get_feature_columns(real_train_df, cfg)
logger.info(
    f"Feature set: {len(feature_columns)} columns derived from real training data."
)

# Fit feature encoders once on the real training set; reuse for the real
# test set and every synthetic training set (LogisticRegression/MLP need
# fully numerical input).
enc, scaler, cat_cols, num_cols = fit_feature_encoders(real_train_df[feature_columns])

X_train_real = apply_feature_encoders(real_train_df[feature_columns], enc, scaler, cat_cols, num_cols)
X_test_real  = apply_feature_encoders(real_test_df[feature_columns],  enc, scaler, cat_cols, num_cols)

present_protected = [a for a in protected_attrs if a in real_train_df.columns]
missing_protected = [a for a in protected_attrs if a not in real_train_df.columns]
if missing_protected:
    logger.warning(
        f"{len(missing_protected)} configured protected attributes not present "
        f"in the data — skipping: {missing_protected}"
    )

split = DataSplit(
    X_train         = X_train_real,
    X_test          = X_test_real,
    y_train         = real_train_df[target_col].astype(int),
    y_test          = real_test_df[target_col].astype(int),
    protected_train = real_train_df[present_protected].copy(),
    protected_test  = real_test_df[present_protected].copy(),
    feature_names   = feature_columns,
    target_name     = target_col,
    protected_attrs = present_protected,
    encoders        = {"ordinal": enc},
    scaler          = scaler,
)

evaluator = Evaluator(split, cfg)
print(f"DataSplit ready. X_train={split.X_train.shape}  X_test={split.X_test.shape}  "
      f"protected={len(present_protected)}")

In [ ]:
classifiers    = list(cfg["experiments"]["classifiers"].keys())
baseline_label = cfg["experiments"]["baseline_label"]

rows = []
per_attr_fairness = {baseline_label: {}, **{m: {} for m in synthetic_datasets}}

# Loop classifier-outer / method-inner so Evaluator.baseline() correctly
# anchors the real-data reference before each classifier's synthetic runs.
for clf_name in classifiers:
    logger.info(f"==> [{baseline_label} | {CLASSIFIER_DISPLAY_NAMES[clf_name]}]")
    result = evaluator.baseline(clf_name)
    result = {**result, "mmd": 0.0}  # MMD(real, real) = 0 by definition
    per_attr_fairness[baseline_label][clf_name] = evaluator.last_fairness_detail
    rows.append({"method": baseline_label, "classifier": clf_name, **result})

    for method, syn_df in synthetic_datasets.items():
        logger.info(f"==> [{method} | {CLASSIFIER_DISPLAY_NAMES[clf_name]}]")

        missing = [c for c in feature_columns if c not in syn_df.columns]
        if missing:
            logger.warning(f"  [{method}] {len(missing)} feature cols absent — filled with 0.")
        X_synth_raw = pd.DataFrame(
            {c: syn_df[c] if c in syn_df.columns else 0.0 for c in feature_columns},
            index=syn_df.index,
        )
        X_synth = apply_feature_encoders(X_synth_raw, enc, scaler, cat_cols, num_cols)
        y_synth = syn_df[target_col].astype(int)

        result = evaluator.evaluate(
            X_synth         = X_synth,
            y_synth         = y_synth,
            classifier_name = clf_name,
            generator_name  = method,
            X_real_for_mmd  = split.X_train,
        )
        per_attr_fairness[method][clf_name] = evaluator.last_fairness_detail
        rows.append({"method": result.pop("generator"), "classifier": result.pop("classifier"), **result})

metrics_df = pd.DataFrame(rows)

# Reorder columns for readability.
id_cols    = ["method", "classifier"]
util_cols  = [
    c for c in ["balanced_accuracy", "f1_macro", "roc_auc", "brier_score", "mmd"]
    if c in metrics_df.columns
]
fair_cols  = [c for c in metrics_df.columns if c.startswith("mean_")]
extra_cols = [c for c in metrics_df.columns if c not in id_cols + util_cols + fair_cols]
metrics_df = metrics_df[id_cols + util_cols + fair_cols + extra_cols]

logger.info(f"Experiment matrix complete. Shape: {metrics_df.shape}")
display(metrics_df)

### 3.1 Save the raw results matrix

The full metrics matrix is persisted as a CSV for downstream use in Notebook 5 (analysis and interpretation) and for reproducibility.

In [ ]:
from src.data.persistence import save_csv


metrics_path = EXPERIMENTS_DIR / "metrics_matrix.csv"
save_csv(metrics_df, metrics_path)
logger.info(f"Metrics matrix saved to: {metrics_path}")

## 4. Delta Analysis

Raw metric values already tell us something, but the key quantity of interest is the **delta**: how much does each metric change when we switch from real to synthetic training data? We define:

$$\Delta m_{\text{method, clf}} = m_{\text{method, clf}} - m_{\text{real, clf}}$$

A negative delta on a metric where higher is better (balanced accuracy, F1, AUC, DI) indicates degradation. A positive delta on a metric where lower is better (Brier Score, DPD, EOD) indicates degradation.

This formulation keeps the baseline classifier-specific: we compare CTGAN-XGBoost against Real-XGBoost, not against Real-LogReg, so that classifier effects do not confound the comparison.

In [ ]:
from src.evaluation.delta import (
    compute_delta_matrix,
    get_utility_delta_columns,
    get_fairness_delta_columns
)


delta_df = compute_delta_matrix(metrics_df, cfg)

# Select only the synthetic rows (exclude the baseline)
baseline_label = cfg["experiments"]["baseline_label"]
delta_syn = delta_df[delta_df["method"] != baseline_label].copy()

util_delta_cols  = get_utility_delta_columns(delta_df)
fair_delta_cols  = get_fairness_delta_columns(delta_df)

logger.info(f"Utility delta columns: {util_delta_cols}")
logger.info(f"Fairness delta columns: {fair_delta_cols}")
display(delta_syn[["method", "classifier"] + util_delta_cols + fair_delta_cols])

### 4.1 Save the delta matrix

In [ ]:
delta_path = EXPERIMENTS_DIR / "delta_matrix.csv"
save_csv(delta_df, delta_path)
logger.info(f"Delta matrix saved to: {delta_path}")

### 4.2 Save per-attribute fairness breakdowns

In addition to the aggregated fairness summary, we save one CSV file per (method, classifier) pair containing the per-protected-attribute fairness breakdown. This granular data is essential for the analysis in Notebook 5, where we investigate which circumstances are most affected by synthetic data generation.

In [ ]:
per_attr_dir = EXPERIMENTS_DIR / "per_attribute_fairness"
ensure_dir(per_attr_dir)

for method, clf_dict in per_attr_fairness.items():
    for clf_name, fair_df in clf_dict.items():
        filename = f"fairness_{method}_{clf_name}.csv"
        save_csv(fair_df, per_attr_dir / filename)

logger.info(f"Per-attribute fairness CSVs saved to: {per_attr_dir}")

## 5. Visualisations

Each figure addresses a specific analytical question:

| Figure | Question |
|--------|----------|
| 5.1 MMD | How faithfully does each method reproduce the real training distribution? |
| 5.2 Utility metrics | What is the absolute utility of each (method, classifier) combination? |
| 5.3 Fairness metrics | What is the absolute fairness of each configuration? |
| 5.4 Delta heat maps | Where are the biggest losses relative to the real baseline? |
| 5.5 Utility vs. fairness scatter | Are utility loss and fairness degradation correlated? |
| 5.6 Per-attribute breakdown | Which protected attributes are most affected by synthetic generation? |

### 5.1 Maximum Mean Discrepancy

MMD measures the distance between the distribution of synthetic training features and the real training features using an RBF kernel (bandwidth set via the median heuristic). A lower MMD indicates that the synthetic data more faithfully reproduces the real data's statistical structure.

MMD is independent of the downstream classifier — `Evaluator.evaluate()` recomputes it per (method, classifier) call for simplicity, but the value is identical across classifiers for a given method since it only depends on the training features.

In [ ]:
from src.utils.plotting import plot_mmd_bar


fig_mmd = plot_mmd_bar(metrics_df, cfg, save=True)
plt.show(); plt.close()

### 5.2 Utility metrics

We plot absolute utility metrics grouped by classifier, with one bar per generation method. The real baseline is always shown first for reference. Balanced accuracy is the primary metric given the class imbalance in the dataset; the remaining metrics (F1-macro, ROC-AUC, Brier Score) provide complementary perspectives on prediction quality.

In [ ]:
from src.utils.plotting import plot_utility_bar


utility_metric_cols = [
    m for m in cfg["experiments"]["utility_metrics"] if m != "mmd" and m in metrics_df.columns
]

for metric in utility_metric_cols:
    fig = plot_utility_bar(metrics_df, metric=metric, cfg=cfg, save=True)
    plt.show(); plt.close()

### 5.3 Fairness metrics

We plot the mean fairness metrics (averaged across all 12 protected attributes) per (method, classifier) combination. Recall the interpretation:

- **DPD** (Demographic Parity Difference): lower is fairer; 0 = perfect parity.
- **EOD** (Equalized Odds Difference): lower is fairer; 0 = equal TPR and FPR across groups.
- **DI** (Disparate Impact ratio): higher is fairer; DI < 0.8 triggers the legal four-fifths rule.

In [ ]:
from src.utils.plotting import plot_fairness_bar


fairness_summary_cols = [c for c in metrics_df.columns if c.startswith("mean_")]

for metric in fairness_summary_cols:
    fig = plot_fairness_bar(metrics_df, metric=metric, cfg=cfg, save=True)
    plt.show(); plt.close()

### 5.4 Delta heatmaps

The heatmaps show the change in each metric relative to the real-data baseline for the same classifier. Red cells indicate degradation; blue cells indicate improvement. By convention, a cell is red when the synthetic model is worse than the real baseline for that metric.

In [ ]:
from src.utils.plotting import plot_delta_heatmap


# Utility delta heatmap
if util_delta_cols:
    fig = plot_delta_heatmap(
        delta_df,
        delta_columns=util_delta_cols,
        filename_suffix="utility",
        cfg=cfg,
        title="Utility metrics (synthetic vs. real baseline, per classifier)",
        save=True
    )
    plt.show(); plt.close()

# Fairness delta heatmap
if fair_delta_cols:
    fig = plot_delta_heatmap(
        delta_df,
        delta_columns=fair_delta_cols,
        filename_suffix="fairness",
        cfg=cfg,
        title="Fairness metrics (synthetic vs. real baseline, per classifier)",
        save=True
    )
    plt.show(); plt.close()

### 5.5 Utility loss vs. Fairness degradation

A central question of this project is whether utility loss and fairness degradation are correlated, i.e. whether the generation methods that lose the most accuracy also lose the most fairness, or whether these dimensions are largely independent. This scatter plot maps each (method, classifier) pair onto the utility–fairness delta plane.

The axes are oriented so that points further up and further left represent *jointly worse* outcomes on both dimensions. Points near the origin are close to the real baseline.

In [ ]:
from src.utils.plotting import plot_utility_fairness_scatter


# Use balanced accuracy (primary utility metric) and mean DPD (primary fairness metric).
util_scatter_col = "delta_balanced_accuracy"
fair_scatter_col = "delta_mean_dpd"

if util_scatter_col in delta_df.columns and fair_scatter_col in delta_df.columns:
    fig = plot_utility_fairness_scatter(
        delta_df,
        utility_col=util_scatter_col,
        fairness_col=fair_scatter_col,
        cfg=cfg,
        save=True
    )
    plt.show(); plt.close()
else:
    logger.warning(
        f"Scatter requires '{util_scatter_col}' and '{fair_scatter_col}'. "
        "Check that balanced_accuracy and mean_dpd are in the metrics config."
    )

### 5.6 Per-attribute fairness breakdown

The aggregated fairness metrics in 5.3 summarise across all 12 protected attributes. Here we drill down to the per-attribute level for a selected (method, classifier) configuration. This is the most policy-relevant view: it identifies *which* groups (e.g., students with low parental education, immigrant students) are most harshly affected when a particular synthetic generation method is used.

We produce one breakdown plot per generation method using the classifier that showed the largest mean fairness degradation.

In [ ]:
from src.utils.plotting import plot_per_attribute_fairness


# For each generation method, pick the classifier with the highest mean DPD
# (i.e. the worst-case fairness configuration) to highlight in the breakdown.

for method in cfg["generation"]["methods"]:
    # Find the classifier with the highest mean DPD for this method.
    method_rows = metrics_df[metrics_df["method"] == method]

    worst_clf = method_rows.loc[method_rows["mean_dpd"].idxmax(), "classifier"]
    fair_df   = per_attr_fairness[method].get(worst_clf)

    for fairness_metric in list(fair_df.columns):
        title = (
            f"{fairness_metric.upper()} per attribute -- "
            f"{method.upper()} / {worst_clf.replace('_', ' ').title()}"
        )
        save_path = (FIGURES_DIR / f"per_attr_{fairness_metric}.png")
        
        fig = plot_per_attribute_fairness(
            fair_df,
            metric    = fairness_metric,
            cfg       = cfg,
            title     = title,
            save = True,
            filename_suffix = f"_{method}_{worst_clf}"
        )
        plt.show(); plt.close()

## 6. Summary Statistics

### 6.1 Utility summary table

Mean utility metrics across classifiers, grouped by generation method. This collapses the classifier dimension to give a method-level picture of utility performance.

In [ ]:
util_cols = [
    m for m in list(metrics_df.columns)
    if m != "mmd"
]

utility_summary = metrics_df.groupby("method")[util_cols].mean().round(4)

# Add MMD column (method-level, not classifier-level).
utility_summary["mmd"] = metrics_df.groupby("method")["mmd"].first().round(4)

display(utility_summary)

save_csv(utility_summary.reset_index(), EXPERIMENTS_DIR / "utility_summary.csv")

### 6.2 Fairness summary table

Mean fairness metrics across classifiers, grouped by generation method.

In [ ]:
fairness_summary_cols = [c for c in metrics_df.columns if c.startswith("mean_")]

fairness_summary = metrics_df.groupby("method")[fairness_summary_cols].mean().round(4)

display(fairness_summary)

save_csv(fairness_summary.reset_index(), EXPERIMENTS_DIR / "fairness_summary.csv")

### 6.3 Delta summary table

Mean deltas relative to the real baseline, grouped by generation method. This is the primary table for the report's Results section, as it directly quantifies the cost of anonymisation in terms of both utility and fairness.

In [ ]:
delta_cols_all = util_delta_cols + fair_delta_cols

if delta_cols_all:
    delta_syn_subset = delta_df[
        (delta_df["method"] != baseline_label)
    ]
    delta_summary = delta_syn_subset.groupby("method")[delta_cols_all].mean().round(4)
    
    display(delta_summary)

    save_csv(delta_summary.reset_index(), EXPERIMENTS_DIR / "delta_summary.csv")

## 7. Outputs Summary

The following artefacts have been produced and saved by this notebook:

| Artefact | Path | Description |
|----------|------|-------------|
| `metrics_matrix.csv` | `results/experiments/` | Raw (method, classifier) × metric matrix |
| `delta_matrix.csv`   | `results/experiments/` | Same, but with Δ columns relative to real baseline |
| `utility_summary.csv`  | `results/experiments/` | Utility metrics averaged across classifiers, per method |
| `fairness_summary.csv` | `results/experiments/` | Fairness metrics averaged across classifiers, per method |
| `delta_summary.csv`    | `results/experiments/` | Mean deltas per method |
| `per_attribute_fairness/` | `results/experiments/` | Per-attribute fairness CSVs (one per config) |
| `*.pdf` figures | `results/figures/experiments/` | All visualisations described in §6 |

These outputs feed directly into **Notebook 5**, where they are interpreted in terms of the project's research questions:
- **Q1**: How much utility is lost under synthetic training data?
- **Q2**: How does synthetic generation alter standard fairness metrics?
- **Q3 (exploratory)**: Is utility loss correlated with fairness degradation?